# 13 从零实现线性回归

本节目标：手动初始化参数、手写模型函数、手写损失函数、手写小批量 SGD，让训练后的参数接近合成数据的真实参数。

In [ ]:
import torch

torch.manual_seed(42)
torch.set_printoptions(precision=4)

X = torch.randn(1000, 2)
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
y = X @ true_w + true_b + torch.randn(1000) * 0.01

print("X shape:", tuple(X.shape))
print("y shape:", tuple(y.shape))
print("true_w:", true_w)
print("true_b:", true_b)

In [ ]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = torch.randperm(num_examples)
    for start in range(0, num_examples, batch_size):
        batch_indices = indices[start:start + batch_size]
        yield features[batch_indices], labels[batch_indices]


def linreg(X, w, b):
    return X @ w + b


def squared_loss(y_hat, y):
    return (y_hat - y) ** 2 / 2


def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [ ]:
w = torch.randn(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.03
batch_size = 32
num_epochs = 8

for epoch in range(num_epochs):
    for X_batch, y_batch in data_iter(batch_size, X, y):
        y_hat = linreg(X_batch, w, b)
        loss = squared_loss(y_hat, y_batch)
        loss.sum().backward()
        sgd([w, b], lr, batch_size)

    with torch.no_grad():
        train_loss = squared_loss(linreg(X, w, b), y).mean()
    print(f"epoch {epoch + 1}, loss {train_loss.item():.8f}")

print("\n真实 w:", true_w)
print("学到的 w:", w.detach())
print("w 误差:", (w.detach() - true_w))
print("真实 b:", true_b)
print("学到的 b:", b.item())
print("b 误差:", b.item() - true_b)

In [ ]:
assert torch.allclose(w.detach(), true_w, atol=0.08)
assert abs(b.item() - true_b) < 0.08
print("参数已经接近真实参数。")

## 小结

从零实现时，我们自己负责所有细节：打乱数据、切 batch、写模型、写损失、调用 `backward()`、更新参数、清空梯度。